In [ ]:
import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
from skimage import measure
import os
import pandas as pd
import scipy.io as sio
import logging

logging.getLogger().setLevel(logging.ERROR)
basepath =  r"X:\data\Barrage\SNCG08\day14"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="mouse"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\complex_system\manifold"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# Create the folder if it doesn't exist
if not os.path.exists(new_folder_path):
    os.makedirs(new_folder_path)
    print(f"Folder created: {new_folder_path}")
else:
    print(f"Folder already exists: {new_folder_path}")
# load in spike data from hpc pyramidal cells
st, cm = loading.load_spikes(
    basepath, brainRegion=brain_region                          # putativeCellType="Pyr"
)
# load theta epochs
state_dict = loading.load_SleepState_states(basepath)
nrem_epochs = nel.EpochArray(
    state_dict["NREMstate"],
)
awake_epochs = nel.EpochArray(
    state_dict["WAKEstate"],
)
# theta_epochs = nel.EpochArray(
#     state_dict["THETA"],
# )
rem_epochs=nel.EpochArray(state_dict["REMstate"],)
try:
    # Try default loader
    epoch_df = loading.load_epoch(basepath)
except Exception as e:
    print(f"load_epoch failed with error: {e}")
    print("Falling back to manual .mat file load...")

    # Load the .mat file
    mat = sio.loadmat(os.path.join(basepath, "DayNight25.MergePoints.events.mat"), squeeze_me=True)

    # Extract the 'MergePoints' structure
    merge_points = mat['MergePoints']

    # Extract the timestamps array: shape (N, 2)
    timestamps = merge_points['timestamps']  # this is the field you're interested in
    # Unwrap the array inside
    timestamps = np.array(timestamps.tolist())

    # Now you should have shape (N, 2)
    epoch_df = pd.DataFrame(timestamps, columns=["startTime", "stopTime"])

sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
E_spikes = st.time[(cm.putativeCellType == 'Pyramidal Cell')]
I_spikes= st.time[(cm.putativeCellType.str.contains("Interneuron"))]
spike_times=st.time
n_cell=len(spike_times)
print(f"number of cells: {n_cell}")

Folder already exists: W:\data\PPP\PPP7\day7\complex_system
number of cells: 249


In [389]:
import numpy as np

# spike_times: list / array of arrays (one per neuron)
spike_times = [np.asarray(st) for st in spike_times if len(st) > 0]

# 1) flatten all spike times
all_spike_times = np.concatenate(spike_times)

# 2) sort (important!)
all_spike_times = np.sort(all_spike_times)

# 3) compute ISI
isi = np.diff(all_spike_times)

# 4) mean ISI
mean_isi = isi.mean()

print("Pooled mean ISI:", mean_isi)



Pooled mean ISI: 0.002863294684169759


In [371]:
import os, re, numpy as np, pandas as pd
import glob  # <-- add this
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False,
            simplify_cells=True
        )
    except TypeError:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)
def _to_1d_array(x):
    """
    Robustly coerce MATLAB-loaded field to a 1D numpy array.
    - Scalars -> 1-length array
    - Lists/tuples/ndarrays -> flattened 1D
    - None / missing -> empty array
    """
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)  # start as object to avoid surprises
    if arr.ndim == 0:
        # scalar or unsized object
        return np.atleast_1d(arr).astype(float, copy=False)
    # flatten and try float cast (ignore non-numeric gracefully)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        # fallback: best-effort, convert elementwise
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    # MATLAB char arrays sometimes come in as numpy arrays
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        # arrays or weird containers -> join characters
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        # e.g., array of characters
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def load_d2_timeline_complete(basepath, region):
    """Complete version that loads ALL sleep-state data, including d2_err."""
    region = region.upper()
    search_dir = os.path.join(basepath, "complex_system")
    pats = [f"*_{region}_D2_time_resolved.mat", f"*_{region}_D2_time.mat"]
    # pats = [f"*_CA1_CA2_D2_time_resolved_error_bar_80ms_60second.mat", f"*_CA1_CA2_D2_time_error_bar_80ms_60second.mat"]
    # pats = [f"*_CA1_D2_time_resolved_error_bar_80ms_60second.mat", f"*_CA1_D2_time_error_bar_80ms_60second.mat"]
    files = []
    for pat in pats:
        files.extend(glob.glob(os.path.join(search_dir, pat)))
    if not files:
        raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")

    mat_path = max(files, key=os.path.getmtime)
    print(f"Loading: {os.path.basename(mat_path)}")

    mat = _load_mat_any(mat_path)
    results = mat['results']
    params = mat.get('params', {})

    # Make results iterable no matter if it's a list, ndarray, or dict
    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

    print(f"📊 Found {len(iter_results)} result items (sleep states)")
    print(f"📊 Parameters: {params}")

    # Try to infer animal/day from filename if available
    fname = os.path.basename(mat_path)
    animal_name = None
    day = None
    m_day = re.search(r'day(\d+)', fname, flags=re.I)
    if m_day:
        day = int(m_day.group(1))
    m_animal = re.match(r'([^_]+)_day\d+_', fname)
    if m_animal:
        animal_name = m_animal.group(1)

    all_centers, all_d2, all_d2err, all_states = [], [], [], []
    n_states_with_err = 0

    for i, result_item in enumerate(iter_results):
        # Access as dict-like regardless of MATLAB struct quirks
        state = _to_state_str(result_item.get('state', 'UNKNOWN')) if hasattr(result_item, 'get') else \
                _to_state_str(result_item['state']) if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) else 'UNKNOWN'

        centers_raw = result_item.get('center', []) if hasattr(result_item, 'get') else \
                      (result_item['center'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'center' in result_item else [])
        d2_raw = result_item.get('d2', []) if hasattr(result_item, 'get') else \
                 (result_item['d2'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2' in result_item else [])
        d2err_raw = result_item.get('d2_err', []) if hasattr(result_item, 'get') else \
                    (result_item['d2_err'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2_err' in result_item else [])

        centers  = _to_1d_array(centers_raw)
        d2_vals  = _to_1d_array(d2_raw)
        d2_err   = _to_1d_array(d2err_raw)

        # Presence flag (some states may not have d2_err)
        has_err = d2_err.size > 0 and np.any(np.isfinite(d2_err))
        n_states_with_err += int(has_err)

        # Harmonize lengths: truncate all to the common min
        sizes = [centers.size, d2_vals.size] + ([d2_err.size] if has_err else [])
        n_pts = min(sizes) if min(sizes) > 0 else 0
        print(f"   State {i}: {state} - requested points={centers.size}, actual used={n_pts}")

        if n_pts == 0:
            print("     ⚠️ Empty state - no data points")
            continue

        centers = centers[:n_pts]
        d2_vals = d2_vals[:n_pts]
        d2_err  = (d2_err[:n_pts] if has_err else np.full(n_pts, np.nan, dtype=float))

        # Ranges
        try:
            print(f"     Time range: {float(np.nanmin(centers))} to {float(np.nanmax(centers))}")
        except ValueError:
            print("     Time range: (all NaN)")

        try:
            print(f"     D2 range: {float(np.nanmin(d2_vals)):.4f} to {float(np.nanmax(d2_vals)):.4f}")
        except ValueError:
            print("     D2 range: (all NaN)")

        if np.any(np.isfinite(d2_err)):
            try:
                dmin, dmax = float(np.nanmin(d2_err)), float(np.nanmax(d2_err))
                print(f"     D2_ERR range: {dmin:.4f} to {dmax:.4f}")
            except ValueError:
                print("     D2_ERR range: (all NaN)")
        else:
            print("     D2_ERR: missing → filled with NaN")

        all_centers.append(centers)
        all_d2.append(d2_vals)
        all_d2err.append(d2_err)
        all_states.append(np.array([state]*n_pts, dtype=object))

    if len(all_centers) == 0:
        raise ValueError("No data found in any sleep state - all states are empty")

    # Concatenate
    all_centers = np.concatenate(all_centers)
    all_d2      = np.concatenate(all_d2)
    all_d2err   = np.concatenate(all_d2err)
    all_states  = np.concatenate(all_states)

    # DataFrame (include d2_err)
    df = pd.DataFrame({
        'center_s': all_centers.astype(float, copy=False),
        'd2':       all_d2.astype(float, copy=False),
        'd2_err':   all_d2err.astype(float, copy=False),
        'state':    all_states.astype(str)
    }).sort_values('center_s').reset_index(drop=True)

    print(f"\n✅ Complete timeline created:")
    print(f"📊 Total data points: {len(df)}")
    print(f"📊 Time range: {df['center_s'].min()} to {df['center_s'].max()}")
    print(f"📊 Duration: {(df['center_s'].max() - df['center_s'].min()):.1f} seconds")
    print(f"📊 State distribution:")
    print(df['state'].value_counts())

    # Empty-state tracker
    empty_states = []
    for ri in iter_results:
        st = _to_state_str(ri.get('state','UNKNOWN')) if hasattr(ri,'get') else \
             _to_state_str(ri['state']) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'state' in ri else 'UNKNOWN'
        csz = np.size(np.atleast_1d(ri.get('center', []))) if hasattr(ri,'get') else \
              np.size(np.atleast_1d(ri['center'])) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'center' in ri else 0
        if csz == 0:
            empty_states.append(st)

    meta = {
        "animal_name": animal_name,
        "day": day,
        "region": region,
        "path": mat_path,
        "params": params,
        "columns": list(df.columns),
        "shape": df.shape,
        "n_states": len(iter_results),
        "states_with_d2_err": n_states_with_err,
        "empty_states": empty_states
    }

    return df, meta



# Test the complete loading function
try:
    ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
    
    # pfc_complete_df, pfc_complete_meta = load_d2_timeline_complete(basepath, "PFC")

    
except Exception as e:
    print(f"❌ Failed: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)

Loading: day20_CA1_D2_time_resolved.mat
📊 Found 3 result items (sleep states)
📊 Parameters: {'window_size': 30, 'step_size': 2, 'guard_sec': 5, 'deltaT': 0.04, 'order': 20, 'quant': 0.1, 'brain_region': 'CA1', 'n_cells': 14, 'total_spikes': 2250385}
   State 0: NREM - requested points=4171, actual used=4171
     Time range: 1595.0 to 29722.0
     D2 range: 0.0558 to 0.2297
     D2_ERR: missing → filled with NaN
   State 1: REM - requested points=294, actual used=294
     Time range: 2861.0 to 29335.0
     D2 range: 0.0458 to 0.1971
     D2_ERR: missing → filled with NaN
   State 2: WAKE - requested points=5038, actual used=5038
     Time range: 67.0 to 26853.0
     D2 range: 0.0259 to 0.2648
     D2_ERR: missing → filled with NaN

✅ Complete timeline created:
📊 Total data points: 9503
📊 Time range: 67.0 to 29722.0
📊 Duration: 29655.0 seconds
📊 State distribution:
state
WAKE    5038
NREM    4171
REM      294
Name: count, dtype: int64



In [372]:
ca1_complete_df

,center_s,d2,d2_err,state
0,67.0,0.104358,NaN,WAKE
1,69.0,0.105953,NaN,WAKE
2,71.0,0.118883,NaN,WAKE
3,73.0,0.135077,NaN,WAKE
4,75.0,0.135944,NaN,WAKE
...,...,...,...,...
9498,29714.0,0.155707,NaN,NREM
9499,29716.0,0.144181,NaN,NREM
9500,29718.0,0.149043,NaN,NREM
9501,29720.0,0.157950,NaN,NREM


In [373]:
epoch_df

,name,startTime,stopTime,environment,manipulation,stimuli,behavioralParadigm,notes,basepath
0,presleep_230108_091019,0.0000,11559.09115,sleepbox,presleep,NaN,NaN,NaN,X:\data\Barrage\NN10\day20
1,stim_230108_124958,11559.0912,18761.15835,sleepbox,swr_stimulation,swr_generation,NaN,NaN,X:\data\Barrage\NN10\day20
2,postsleep_230108_145008,18761.1584,29868.76155,sleepbox,postsleep,NaN,NaN,NaN,X:\data\Barrage\NN10\day20


SWR and barrage

In [374]:
import numpy as np
import scipy.io as sio      
import scipy
# Load ripple events
ripple_events = loading.load_ripples_events(basepath=basepath)
ripple_epochs = nel.EpochArray(np.array([ripple_events.start, ripple_events.stop]).T)
ripple_peaks=ripple_events.peaks
    # Load barrage events
ripple_epochs=nrem_epochs&ripple_epochs
try:
    # Load barrage events
    #brpath=r"W:\data\PPP\CA2\PPP7\day8"
    barrage_events = loading.load_barrage_events(basepath=basepath)  
    # Create barrage epochs
    barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
    # Find barrages in NREM epochs
    barrages_in_nrem = barrage_epochs & nrem_epochs
    mat_path = os.path.join(basepath, f'{basename}.HSEn2.events.mat')

    data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    barrage = data['HSEn2']
    barrage_timestamps= barrage.timestamps 
    barrage_epochs=nrem_epochs&barrage_epochs
except Exception as e:
    # General exception for unexpected errors
    print(f"No barrage file: {e}")
    barrage_epochs=[]
    # Extract start times


No barrage file: 'DataFrame' object has no attribute 'start'


D:\Documents\GitHub\neuro_py\neuro_py\io\loading.py:1036: UserWarning: No barrage file found for X:\data\Barrage\NN10\day20


d2 vs manipulation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
# --- 1) Define helper function to get D2 values in a given epoch ---
def get_d2_in_epoch(df, epoch_row, state='NREM'):
    mask = (
        (df['center_s'] >= epoch_row['startTime']) &
        (df['center_s'] < epoch_row['stopTime']) &
        (df['state'].astype(str).str.upper() == state)
    )
    return df.loc[mask, 'd2'].values

# --- 2) Find presleep and postsleep epochs ---
presleep_epochs = epoch_df[epoch_df['name'].str.contains('presleep', case=False, na=False)]
postsleep_epochs = epoch_df[epoch_df['name'].str.contains('postsleep', case=False, na=False)]

# --- 3) Extract D² values (NREM only) ---
presleep_d2 = np.concatenate([get_d2_in_epoch(ca1_complete_df, row) for _, row in presleep_epochs.iterrows()]) \
              if not presleep_epochs.empty else np.array([])

postsleep_d2 = np.concatenate([get_d2_in_epoch(ca1_complete_df, row) for _, row in postsleep_epochs.iterrows()]) \
               if not postsleep_epochs.empty else np.array([])

print(f"Presleep NREM samples: {len(presleep_d2)}")
print(f"Postsleep NREM samples: {len(postsleep_d2)}")

# --- 4) Equalize N by truncating to the first chunk of the shorter one ---
min_len = min(len(presleep_d2), len(postsleep_d2))
presleep_d2 = presleep_d2[:min_len]
postsleep_d2 = postsleep_d2[:min_len]
print(f"Equalized to {min_len} samples per epoch")

# --- 5) Make dataframe for boxplot ---
plot_df = pd.DataFrame({
    'epoch': ['presleep'] * len(presleep_d2) + ['postsleep'] * len(postsleep_d2),
    'd2': np.concatenate([presleep_d2, postsleep_d2])
})

order = ['presleep', 'postsleep']

# make 'epoch' an ordered categorical
plot_df['epoch'] = pd.Categorical(plot_df['epoch'], categories=order, ordered=True)

# now boxplot respects the order
ax = plot_df.boxplot(column='d2', by='epoch', grid=False, showfliers=False)
plt.title('D² in NREM (Presleep vs Postsleep) — equal length')
plt.suptitle('')
plt.ylabel('D²')
plt.tight_layout()
plt.show()
outdir = r"D:\Documents\GitHub\Sleep_criticaltiy\data\ripple_barr\manipulation\sncg"

# --- 1) Ensure output directory exists ---
os.makedirs(outdir, exist_ok=True)

# --- 2) Define output path ---
filename = f"{animal_name}_{day}.pkl"
outpath = os.path.join(outdir, filename)

# --- 3) Save plot_df ---
with open(outpath, "wb") as f:
    pickle.dump(plot_df, f)

print(f"✅ Saved plot_df to: {outpath}")